# Lecture 8: Likelihood Modeling, Bootstrap, and Jackknife Uncertainty

**Unit 2, September 22, 2026**

Lecture 6 introduced nonlinear fitting with `curve_fit`, `least_squares`, and minimizers. Lecture 7 asked whether a fitted model was statistically compatible with data using $\chi^2$, $\Delta\chi^2$, and hypothesis tests.

Today we move from Gaussian least-squares models to **Poisson likelihood models** for event counts. The example is a small, event-wise CMS Open Data dimuon sample. The physics goal is modest: model the event intensity near the $J/\psi\to\mu^+\mu^-$ mass peak, compare a signal-plus-background model with a background-only model, and estimate uncertainties with covariance, bootstrap, and jackknife methods.

This is a teaching analysis, not a publication-quality CMS result.

## 80-minute flow

| Time | Mode | Focus |
| --- | --- | --- |
| 0-8 min | Instructor | From Lecture 7 $\chi^2$ tests to Poisson event likelihoods |
| 8-18 min | Instructor + discussion | CMS dimuon data and event intensity $\lambda(m,p_T)$ |
| 18-30 min | Groups | Activity 1: inspect the event-wise CSV and selected region |
| 30-44 min | Instructor | Extended Poisson likelihood and signal-plus-background model |
| 44-56 min | Groups | Activity 2: complete likelihood-ratio model comparison |
| 56-64 min | Instructor | Covariance from likelihood curvature and its limitations |
| 64-74 min | Groups | Activity 3: summarize bootstrap uncertainty |
| 74-80 min | Instructor + discussion | Jackknife influence diagnostics and takeaways |

Interactive coding time: about 34 minutes, or 43% of class.

## Learning goals

By the end of this lecture you should be able to:

- explain what an event intensity $\lambda(x;\theta)$ means,
- write an extended Poisson log-likelihood for event-wise data,
- fit a signal-plus-background intensity model with `scipy.optimize.minimize`,
- compare nested models using a likelihood-ratio test,
- estimate parameter uncertainties with likelihood curvature, bootstrap resampling, and jackknife diagnostics,
- describe when bootstrap or jackknife methods reveal information not obvious from a covariance matrix.

# Part 1: Public CMS dimuon data

We use a reduced CSV derived from the CMS Run2010B dimuon open-data sample. The original public record is:

> Thomas McCauley, **Dimuon event information derived from the Run2010B public Mu dataset**, CERN Open Data Portal (2014), DOI: [10.7483/OPENDATA.CMS.CB8H.MFFA](https://doi.org/10.7483/OPENDATA.CMS.CB8H.MFFA).

The repository CSV used here is `data/cms_dimuon_jpsi_3000.csv`. It contains 3000 opposite-sign dimuon events selected from the invariant-mass region

$$
2.6 < m_{\mu\mu} < 3.6\ \mathrm{GeV},
$$

with a dimuon transverse momentum column constructed from the two muon momenta. The original source contains many more events; this reduced version is intentionally small enough to keep in the GitHub repository and fast enough for in-class fitting.

The two modeled event coordinates are:

$$
x=(m,p_T),
$$

where $m$ is the dimuon invariant mass and $p_T$ is the transverse momentum of the dimuon pair.

In [ ]:
%matplotlib inline

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import erf
from scipy.stats import chi2

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

rng = np.random.default_rng(20260922)
print(f"NumPy {np.__version__} | pandas {pd.__version__}")

In [ ]:
DATA_PATH_CANDIDATES = [
    Path("../data/cms_dimuon_jpsi_3000.csv"),
    Path("data/cms_dimuon_jpsi_3000.csv"),
]
DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), DATA_PATH_CANDIDATES[0])
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find the reduced CMS dimuon CSV. Tried: {DATA_PATH_CANDIDATES}")

cms_data = pd.read_csv(DATA_PATH)
analysis_data = cms_data.copy()
analysis_data.head()

### Discussion question 1

In Lecture 7, a model predicted a value $y_i$ with uncertainty $\sigma_i$ at each measured point. Here, each row is an **event**. What should a model predict now: a value for each event, a probability density, or an expected number of events?

### In-class coding activity 1: inspect the event-wise data, 12 minutes

Complete the quick inspection below. The goal is to identify the analysis region and the two event coordinates that will enter the likelihood.

In [ ]:
# TODO: Choose the columns that define the two-dimensional event space.
mass_column = ...  # TODO: "dimuon_mass_GeV"
pt_column = ...    # TODO: "dimuon_pt_GeV"

# TODO: Compute the number of events and the mass/pt ranges.
number_of_events = ...
mass_range_from_data = (..., ...)  # TODO: min and max mass.
pt_range_from_data = (..., ...)    # TODO: min and max dimuon pT.

pd.DataFrame([
    {
        "number_of_events": number_of_events,
        "mass_min_GeV": mass_range_from_data[0],
        "mass_max_GeV": mass_range_from_data[1],
        "pt_min_GeV": pt_range_from_data[0],
        "pt_max_GeV": pt_range_from_data[1],
    }
])

In [ ]:
MASS_RANGE = (2.6, 3.6)
PT_RANGE = (0.0, 30.0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(analysis_data["dimuon_mass_GeV"], bins=60, histtype="step", lw=1.8)
axes[0].set_xlabel(r"$m_{\mu\mu}$ (GeV)")
axes[0].set_ylabel("events")
axes[0].set_title("Dimuon mass near the J/psi region")

axes[1].hist(analysis_data["dimuon_pt_GeV"], bins=50, histtype="step", lw=1.8)
axes[1].set_xlabel(r"dimuon $p_T$ (GeV)")
axes[1].set_ylabel("events")
axes[1].set_title("Dimuon transverse momentum")
fig.tight_layout()

# Part 2: From Gaussian $\chi^2$ to Poisson likelihoods

Lecture 7 used a Gaussian measurement model for repeated measurements of a quantity $y_i$:

$$
y_i \sim \mathcal{N}\left(f(x_i;\theta),\sigma_i^2\right).
$$

That assumption leads to

$$
-2\ln L(\theta)=\chi^2(\theta)+\mathrm{constant}.
$$

Today the data are different. Each row in the CMS dimuon table is an **event** with coordinates

$$
x=(m,p_T).
$$

Instead of predicting a measured value $y_i$ at a fixed location, the model predicts an **event intensity**

$$
\lambda(x;\theta),
$$

with units of expected events per unit event-space volume. Therefore

$$
\Lambda(\theta)=\int_\Omega \lambda(x;\theta)\,dx
$$

is the expected total number of events in the analysis region $\Omega$.

There are three related likelihoods worth distinguishing.

**1. Gaussian least-squares likelihood.** Use this when the measured quantities are approximately Gaussian distributed around model predictions with known standard uncertainties:

$$
-2\ln L = \sum_i \left[\frac{y_i-f(x_i;\theta)}{\sigma_i}\right]^2 + \mathrm{constant}.
$$

**2. Binned Poisson likelihood.** If the data are counts $n_b$ in bins and the model predicts expected bin counts $\nu_b(\theta)$, use

$$
\ln L_\mathrm{binned}=\sum_b \left[n_b\ln\nu_b(\theta)-\nu_b(\theta)-\ln(n_b!)\right].
$$

This is often the right model for histograms, especially when counts are small enough that Gaussian approximations to bin errors are questionable.

**3. Extended unbinned Poisson likelihood.** If we keep each event coordinate rather than only a histogram bin, an inhomogeneous Poisson point process gives

$$
\ln L_\mathrm{extended}(\theta) =
-\int_\Omega \lambda(x;\theta)\,dx
+ \sum_{i=1}^{N_\mathrm{obs}} \ln\lambda(x_i;\theta)
+ \mathrm{constant}.
$$

The first term is the Poisson penalty for predicting too many or too few total events. The sum rewards models that place high intensity where the observed events actually occurred. The extended unbinned likelihood is closely related to the small-bin limit of the binned Poisson likelihood, but it preserves the event coordinates instead of compressing them into bin counts.

The practical lesson is the same as in the Gaussian case: choose the objective function by writing down the measurement model first.


# Part 3: A two-dimensional signal-plus-background intensity model

We model the intensity in $(m,p_T)$ as a mixture:

$$
\lambda(m,p_T;\theta) =
N_s f_s(m,p_T;\theta_s) + N_b f_b(m,p_T;\theta_b).
$$

The signal model is a truncated Gaussian in mass times an exponential in dimuon $p_T$:

$$
f_s(m,p_T) = G_\mathrm{trunc}(m;\mu,\sigma)\,E_\mathrm{trunc}(p_T;\beta_s).
$$

The background model is exponential in both coordinates:

$$
f_b(m,p_T)=E_\mathrm{trunc}(m;\alpha_b)\,E_\mathrm{trunc}(p_T;\beta_b).
$$

Each component density is normalized over the selected region, so

$$
\int_\Omega f_s\,dm\,dp_T=1,\qquad \int_\Omega f_b\,dm\,dp_T=1,
$$

and therefore

$$
\int_\Omega \lambda\,dm\,dp_T = N_s+N_b.
$$

The main physics quantity in this lecture is the signal yield or peak position. The background yield and background-shape parameters are examples of **nuisance parameters**: they are not the final physics result, but they must be included because they affect the fitted signal. In a realistic analysis, nuisance parameters might also describe calibration, mass resolution, efficiency, or luminosity.


In [ ]:
def normal_cdf(x, mean, sigma):
    return 0.5 * (1.0 + erf((x - mean) / (np.sqrt(2.0) * sigma)))


def truncated_gaussian_pdf(x, mean, sigma, lower, upper):
    normalization = normal_cdf(upper, mean, sigma) - normal_cdf(lower, mean, sigma)
    density = np.exp(-0.5 * ((x - mean) / sigma) ** 2) / (np.sqrt(2.0 * np.pi) * sigma)
    return density / normalization


def truncated_exponential_pdf(x, slope, lower, upper):
    width = upper - lower
    if abs(slope) < 1.0e-10:
        return np.ones_like(x, dtype=float) / width
    normalization = (1.0 - np.exp(-slope * width)) / slope
    return np.exp(-slope * (x - lower)) / normalization


def unpack_full_parameters(transformed_parameters):
    """Use transformed parameters so positive quantities stay positive during fitting."""
    log_signal_yield, log_background_yield, mass_mean, log_mass_sigma, log_mass_slope, log_signal_pt_slope, log_background_pt_slope = transformed_parameters
    return pd.Series(
        {
            "signal_yield": np.exp(log_signal_yield),
            "background_yield": np.exp(log_background_yield),
            "mass_mean_GeV": mass_mean,
            "mass_sigma_GeV": np.exp(log_mass_sigma),
            "background_mass_slope": np.exp(log_mass_slope),
            "signal_pt_slope": np.exp(log_signal_pt_slope),
            "background_pt_slope": np.exp(log_background_pt_slope),
        }
    )


def signal_density(data, parameters):
    mass_density = truncated_gaussian_pdf(
        data["dimuon_mass_GeV"],
        parameters["mass_mean_GeV"],
        parameters["mass_sigma_GeV"],
        *MASS_RANGE,
    )
    pt_density = truncated_exponential_pdf(
        data["dimuon_pt_GeV"],
        parameters["signal_pt_slope"],
        *PT_RANGE,
    )
    return mass_density * pt_density


def background_density(data, parameters):
    mass_density = truncated_exponential_pdf(
        data["dimuon_mass_GeV"],
        parameters["background_mass_slope"],
        *MASS_RANGE,
    )
    pt_density = truncated_exponential_pdf(
        data["dimuon_pt_GeV"],
        parameters["background_pt_slope"],
        *PT_RANGE,
    )
    return mass_density * pt_density


def full_intensity(data, transformed_parameters):
    parameters = unpack_full_parameters(transformed_parameters)
    return (
        parameters["signal_yield"] * signal_density(data, parameters)
        + parameters["background_yield"] * background_density(data, parameters)
    )


def negative_log_likelihood_full(transformed_parameters, data=analysis_data):
    parameters = unpack_full_parameters(transformed_parameters)
    if not (MASS_RANGE[0] < parameters["mass_mean_GeV"] < MASS_RANGE[1]):
        return 1.0e30
    if not (0.005 < parameters["mass_sigma_GeV"] < 0.30):
        return 1.0e30

    event_intensity = full_intensity(data, transformed_parameters)
    if np.any(event_intensity <= 0.0) or np.any(~np.isfinite(event_intensity)):
        return 1.0e30

    expected_events = parameters["signal_yield"] + parameters["background_yield"]
    return expected_events - np.sum(np.log(event_intensity))

In [ ]:
initial_full_parameters = np.array([
    np.log(1600.0),  # signal yield
    np.log(1400.0),  # background yield
    3.09,            # J/psi mass mean, GeV
    np.log(0.035),   # mass width, GeV
    np.log(0.7),     # background mass slope
    np.log(0.06),    # signal pT slope
    np.log(0.07),    # background pT slope
])

full_parameter_bounds = [
    (np.log(1.0), np.log(10000.0)),
    (np.log(1.0), np.log(10000.0)),
    (3.0, 3.2),
    (np.log(0.010), np.log(0.20)),
    (np.log(0.001), np.log(20.0)),
    (np.log(0.001), np.log(2.0)),
    (np.log(0.001), np.log(2.0)),
]

full_fit_result = minimize(
    negative_log_likelihood_full,
    x0=initial_full_parameters,
    method="L-BFGS-B",
    bounds=full_parameter_bounds,
    options={"maxiter": 20000, "ftol": 1.0e-9},
)

full_fit_parameters = unpack_full_parameters(full_fit_result.x)
print("fit success:", full_fit_result.success)
print("negative log-likelihood:", full_fit_result.fun)
full_fit_parameters.to_frame("estimate")

In [ ]:
def mass_projection_density(mass_values, parameters):
    mass_frame = pd.DataFrame({"dimuon_mass_GeV": mass_values, "dimuon_pt_GeV": np.zeros_like(mass_values)})
    signal_mass = truncated_gaussian_pdf(mass_frame["dimuon_mass_GeV"], parameters["mass_mean_GeV"], parameters["mass_sigma_GeV"], *MASS_RANGE)
    background_mass = truncated_exponential_pdf(mass_frame["dimuon_mass_GeV"], parameters["background_mass_slope"], *MASS_RANGE)
    return parameters["signal_yield"] * signal_mass + parameters["background_yield"] * background_mass


def pt_projection_density(pt_values, parameters):
    pt_frame = pd.DataFrame({"dimuon_mass_GeV": np.zeros_like(pt_values), "dimuon_pt_GeV": pt_values})
    signal_pt = truncated_exponential_pdf(pt_frame["dimuon_pt_GeV"], parameters["signal_pt_slope"], *PT_RANGE)
    background_pt = truncated_exponential_pdf(pt_frame["dimuon_pt_GeV"], parameters["background_pt_slope"], *PT_RANGE)
    return parameters["signal_yield"] * signal_pt + parameters["background_yield"] * background_pt

mass_grid = pd.DataFrame({"dimuon_mass_GeV": np.linspace(*MASS_RANGE, 400)})
pt_grid = pd.DataFrame({"dimuon_pt_GeV": np.linspace(*PT_RANGE, 400)})
mass_grid["fit_events_per_GeV"] = mass_projection_density(mass_grid["dimuon_mass_GeV"], full_fit_parameters)
pt_grid["fit_events_per_GeV"] = pt_projection_density(pt_grid["dimuon_pt_GeV"], full_fit_parameters)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

mass_counts, mass_edges, _ = axes[0].hist(analysis_data["dimuon_mass_GeV"], bins=60, histtype="step", lw=1.8, label="data")
mass_bin_width = mass_edges[1] - mass_edges[0]
axes[0].plot(mass_grid["dimuon_mass_GeV"], mass_grid["fit_events_per_GeV"] * mass_bin_width, label="extended-likelihood fit")
axes[0].set_xlabel(r"$m_{\mu\mu}$ (GeV)")
axes[0].set_ylabel("events / bin")
axes[0].legend()

pt_counts, pt_edges, _ = axes[1].hist(analysis_data["dimuon_pt_GeV"], bins=50, histtype="step", lw=1.8, label="data")
pt_bin_width = pt_edges[1] - pt_edges[0]
axes[1].plot(pt_grid["dimuon_pt_GeV"], pt_grid["fit_events_per_GeV"] * pt_bin_width, label="extended-likelihood fit")
axes[1].set_xlabel(r"dimuon $p_T$ (GeV)")
axes[1].set_ylabel("events / bin")
axes[1].legend()
fig.tight_layout()

### Discussion question 2

The fitted yields satisfy $N_s+N_b\approx N_\mathrm{observed}$. Is that a coincidence, a constraint we imposed by hand, or a consequence of the extended likelihood?

# Part 4: Likelihood-ratio model comparison

To test whether the data prefer a signal component, compare the full model to a background-only model. The likelihood-ratio statistic is

$$
q = -2\ln\frac{L_\mathrm{background}}{L_\mathrm{full}}
= 2\left[(-\ln L)_\mathrm{background} - (-\ln L)_\mathrm{full}\right].
$$

For regular nested models, Wilks' theorem says $q$ is approximately distributed as $\chi^2_{\Delta k}$. In this case, the signal yield is bounded below by zero, so the theorem is not exact. Still, $q$ is a useful diagnostic for how strongly the additional signal component improves the likelihood.

In [ ]:
def unpack_background_parameters(transformed_parameters):
    log_background_yield, log_mass_slope, log_pt_slope = transformed_parameters
    return pd.Series(
        {
            "background_yield": np.exp(log_background_yield),
            "background_mass_slope": np.exp(log_mass_slope),
            "background_pt_slope": np.exp(log_pt_slope),
        }
    )


def background_only_intensity(data, transformed_parameters):
    parameters = unpack_background_parameters(transformed_parameters)
    mass_density = truncated_exponential_pdf(data["dimuon_mass_GeV"], parameters["background_mass_slope"], *MASS_RANGE)
    pt_density = truncated_exponential_pdf(data["dimuon_pt_GeV"], parameters["background_pt_slope"], *PT_RANGE)
    return parameters["background_yield"] * mass_density * pt_density


def negative_log_likelihood_background(transformed_parameters, data=analysis_data):
    parameters = unpack_background_parameters(transformed_parameters)
    event_intensity = background_only_intensity(data, transformed_parameters)
    if np.any(event_intensity <= 0.0) or np.any(~np.isfinite(event_intensity)):
        return 1.0e30
    return parameters["background_yield"] - np.sum(np.log(event_intensity))

background_initial_parameters = np.array([np.log(len(analysis_data)), np.log(0.5), np.log(0.07)])
background_bounds = [(np.log(1.0), np.log(10000.0)), (np.log(0.001), np.log(20.0)), (np.log(0.001), np.log(2.0))]

background_fit_result = minimize(
    negative_log_likelihood_background,
    x0=background_initial_parameters,
    method="L-BFGS-B",
    bounds=background_bounds,
    options={"maxiter": 20000, "ftol": 1.0e-9},
)

background_fit_parameters = unpack_background_parameters(background_fit_result.x)
print("background fit success:", background_fit_result.success)
background_fit_parameters.to_frame("estimate")

### In-class coding activity 2: likelihood-ratio test, 12 minutes

Complete the likelihood-ratio comparison between the background-only and signal-plus-background models.

In [ ]:
# TODO: Compute q = 2 * (NLL_background - NLL_full).
likelihood_ratio_statistic = ...

# TODO: Count the extra free parameters in the full model compared with the background-only model.
delta_parameters = ...

# TODO: Use chi2.sf(q, delta_parameters) as the approximate Wilks-theorem p-value.
approximate_p_value = ...

likelihood_ratio_table = pd.DataFrame([
    {
        "comparison": "signal + background vs background only",
        "q_delta_minus_2_log_L": likelihood_ratio_statistic,
        "delta_parameters": delta_parameters,
        "approximate_p_value": approximate_p_value,
        "caveat": "signal yield is bounded at zero; Wilks approximation is not exact",
    }
])
likelihood_ratio_table

In [ ]:
provided_lrt_statistic = 2.0 * (background_fit_result.fun - full_fit_result.fun)
provided_lrt_table = pd.DataFrame([
    {
        "model": "background only",
        "negative_log_likelihood": background_fit_result.fun,
        "n_parameters": 3,
    },
    {
        "model": "signal + background",
        "negative_log_likelihood": full_fit_result.fun,
        "n_parameters": 7,
    },
])
provided_lrt_table["q_vs_background"] = [np.nan, provided_lrt_statistic]
provided_lrt_table

# Part 5: Covariance from likelihood curvature

Near the maximum likelihood estimate, the negative log-likelihood can often be approximated by a quadratic:

$$
-\ln L(\theta) \approx -\ln L(\hat\theta) + \frac{1}{2}(\theta-\hat\theta)^T H (\theta-\hat\theta).
$$

The inverse Hessian $H^{-1}$ estimates the covariance matrix in the fitted parameter coordinates. Here we compute a simple finite-difference Hessian in the transformed parameter coordinates and propagate it back to the physical parameters.

A complementary likelihood-based tool is a **profile likelihood**. Choose a parameter of interest, such as a signal yield $\mu$, and refit all nuisance parameters $\eta$ at each fixed value of $\mu$:

$$
q(\mu)= -2\ln\frac{L(\mu,\hat{\hat{\eta}}_\mu)}{L(\hat{\mu},\hat{\eta})}.
$$

Here $\hat{\mu},\hat{\eta}$ are the global best-fit values, while $\hat{\hat{\eta}}_\mu$ means “the best nuisance parameters when $\mu$ is held fixed.” Profile likelihoods are often more reliable than reading one diagonal element of a covariance matrix, especially when nuisance parameters are correlated with the parameter of interest.


In [ ]:
def numerical_hessian(function, x, step=1.0e-3):
    x = np.asarray(x, dtype=float)
    n_parameters = len(x)
    hessian = np.zeros((n_parameters, n_parameters))
    f0 = function(x)

    for row in range(n_parameters):
        row_step = np.zeros(n_parameters)
        row_step[row] = step
        hessian[row, row] = (function(x + row_step) - 2.0 * f0 + function(x - row_step)) / step**2

        for col in range(row + 1, n_parameters):
            col_step = np.zeros(n_parameters)
            col_step[col] = step
            hessian[row, col] = hessian[col, row] = (
                function(x + row_step + col_step)
                - function(x + row_step - col_step)
                - function(x - row_step + col_step)
                + function(x - row_step - col_step)
            ) / (4.0 * step**2)

    return hessian

transformed_hessian = numerical_hessian(negative_log_likelihood_full, full_fit_result.x)
transformed_covariance = np.linalg.inv(transformed_hessian)

# Convert covariance from transformed coordinates to physical coordinates.
physical_values = full_fit_parameters.to_numpy()
transformation_jacobian = np.diag([
    physical_values[0],  # d exp(log Ns) / d log Ns
    physical_values[1],  # d exp(log Nb) / d log Nb
    1.0,                 # mass mean is not log-transformed
    physical_values[3],
    physical_values[4],
    physical_values[5],
    physical_values[6],
])
physical_covariance = transformation_jacobian @ transformed_covariance @ transformation_jacobian.T
covariance_uncertainties = np.sqrt(np.diag(physical_covariance))

covariance_summary = full_fit_parameters.reset_index()
covariance_summary.columns = ["parameter", "estimate"]
covariance_summary["covariance_uncertainty"] = covariance_uncertainties
covariance_summary

# Part 6: Parametric bootstrap

A parametric bootstrap repeats the experiment under the fitted model:

1. Draw a total number of events from $\mathrm{Poisson}(N_s+N_b)$.
2. Assign each event to signal or background with probabilities proportional to $N_s$ and $N_b$.
3. Draw $(m,p_T)$ values from the fitted signal or background densities.
4. Refit each fake dataset.

The spread of refitted parameters estimates the uncertainty implied by the full simulation of the fitted measurement model.

In [ ]:
def sample_truncated_exponential(size, slope, lower, upper, rng):
    width = upper - lower
    uniform = rng.uniform(size=size)
    if abs(slope) < 1.0e-10:
        return lower + width * uniform
    return lower - np.log(1.0 - uniform * (1.0 - np.exp(-slope * width))) / slope


def sample_truncated_gaussian(size, mean, sigma, lower, upper, rng):
    samples = []
    while len(samples) < size:
        proposal = rng.normal(mean, sigma, size=max(100, 2 * (size - len(samples))))
        accepted = proposal[(proposal >= lower) & (proposal <= upper)]
        samples.extend(accepted.tolist())
    return np.array(samples[:size])


def simulate_from_full_model(parameters, rng):
    expected_events = parameters["signal_yield"] + parameters["background_yield"]
    n_events = rng.poisson(expected_events)
    signal_probability = parameters["signal_yield"] / expected_events
    is_signal = rng.uniform(size=n_events) < signal_probability

    simulated_mass = np.empty(n_events)
    simulated_pt = np.empty(n_events)

    n_signal = int(is_signal.sum())
    n_background = n_events - n_signal

    simulated_mass[is_signal] = sample_truncated_gaussian(
        n_signal,
        parameters["mass_mean_GeV"],
        parameters["mass_sigma_GeV"],
        *MASS_RANGE,
        rng,
    )
    simulated_pt[is_signal] = sample_truncated_exponential(
        n_signal,
        parameters["signal_pt_slope"],
        *PT_RANGE,
        rng,
    )

    simulated_mass[~is_signal] = sample_truncated_exponential(
        n_background,
        parameters["background_mass_slope"],
        *MASS_RANGE,
        rng,
    )
    simulated_pt[~is_signal] = sample_truncated_exponential(
        n_background,
        parameters["background_pt_slope"],
        *PT_RANGE,
        rng,
    )

    return pd.DataFrame({"dimuon_mass_GeV": simulated_mass, "dimuon_pt_GeV": simulated_pt})


def fit_full_model(data, starting_point=full_fit_result.x):
    return minimize(
        negative_log_likelihood_full,
        x0=starting_point,
        args=(data,),
        method="L-BFGS-B",
        bounds=full_parameter_bounds,
        options={"maxiter": 20000, "ftol": 1.0e-9},
    )

bootstrap_rows = []
number_of_bootstrap_samples = 80

for sample_index in range(number_of_bootstrap_samples):
    bootstrap_data = simulate_from_full_model(full_fit_parameters, rng)
    bootstrap_fit = fit_full_model(bootstrap_data)
    if not bootstrap_fit.success:
        continue
    bootstrap_parameters = unpack_full_parameters(bootstrap_fit.x)
    bootstrap_rows.append({"sample": sample_index, **bootstrap_parameters.to_dict()})

bootstrap_results = pd.DataFrame(bootstrap_rows)
bootstrap_results.head()

### In-class coding activity 3: summarize bootstrap uncertainty, 10 minutes

Complete the bootstrap summary for the signal yield, mass mean, and mass width.

In [ ]:
# TODO: Choose the parameters to summarize from bootstrap_results.
bootstrap_columns_to_summarize = ...

bootstrap_summary_rows = []
for column in bootstrap_columns_to_summarize:
    values = bootstrap_results[column]
    bootstrap_summary_rows.append(
        {
            "parameter": column,
            "bootstrap_mean": ...,  # TODO: mean of bootstrap values.
            "bootstrap_std": ...,   # TODO: standard deviation of bootstrap values.
            "p16": ...,             # TODO: 16th percentile.
            "p84": ...,             # TODO: 84th percentile.
        }
    )

bootstrap_summary = pd.DataFrame(bootstrap_summary_rows)
bootstrap_summary

In [ ]:
bootstrap_parameters_to_summarize = ["signal_yield", "mass_mean_GeV", "mass_sigma_GeV"]

provided_bootstrap_summary_rows = []
for parameter_name in bootstrap_parameters_to_summarize:
    bootstrap_values = bootstrap_results[parameter_name]
    provided_bootstrap_summary_rows.append(
        {
            "parameter": parameter_name,
            "bootstrap_mean": bootstrap_values.mean(),
            "bootstrap_std": bootstrap_values.std(ddof=1),
            "p16": np.percentile(bootstrap_values, 16),
            "p84": np.percentile(bootstrap_values, 84),
        }
    )

provided_bootstrap_summary = pd.DataFrame(provided_bootstrap_summary_rows)
provided_bootstrap_summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for axis, column in zip(axes, ["signal_yield", "mass_mean_GeV", "mass_sigma_GeV"]):
    axis.hist(bootstrap_results[column], bins=20, alpha=0.75)
    axis.axvline(full_fit_parameters[column], color="black", lw=2, label="fit")
    axis.set_xlabel(column)
    axis.set_ylabel("bootstrap samples")
    axis.legend(fontsize=8)
fig.tight_layout()

# Part 7: Jackknife influence diagnostics

A full leave-one-event-out jackknife would require 3000 refits, which is too slow for class. Instead, use a **leave-one-mass-bin-out** diagnostic: remove one mass slice at a time, refit, and see which regions most affect the signal yield or peak position.

In [ ]:
mass_bin_edges = np.linspace(*MASS_RANGE, 21)
jackknife_rows = []

for bin_index in range(len(mass_bin_edges) - 1):
    bin_low = mass_bin_edges[bin_index]
    bin_high = mass_bin_edges[bin_index + 1]
    keep_mask = ~analysis_data["dimuon_mass_GeV"].between(bin_low, bin_high, inclusive="left")
    jackknife_data = analysis_data.loc[keep_mask].reset_index(drop=True)
    jackknife_fit = fit_full_model(jackknife_data)
    if not jackknife_fit.success:
        continue
    jackknife_parameters = unpack_full_parameters(jackknife_fit.x)
    jackknife_rows.append(
        {
            "omitted_bin_low": bin_low,
            "omitted_bin_high": bin_high,
            "omitted_bin_center": 0.5 * (bin_low + bin_high),
            "n_omitted": int((~keep_mask).sum()),
            **jackknife_parameters.to_dict(),
        }
    )

jackknife_results = pd.DataFrame(jackknife_rows)
jackknife_results["signal_yield_shift"] = jackknife_results["signal_yield"] - full_fit_parameters["signal_yield"]
jackknife_results["mass_mean_shift"] = jackknife_results["mass_mean_GeV"] - full_fit_parameters["mass_mean_GeV"]
jackknife_results.head()

In [ ]:
most_influential_bins = jackknife_results.reindex(
    jackknife_results["signal_yield_shift"].abs().sort_values(ascending=False).index
)
most_influential_bins[["omitted_bin_low", "omitted_bin_high", "n_omitted", "signal_yield_shift", "mass_mean_shift"]].head()

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 6))
axes[0].axhline(0.0, color="0.5", lw=1)
axes[0].plot(jackknife_results["omitted_bin_center"], jackknife_results["signal_yield_shift"], "o-")
axes[0].set_ylabel("signal yield shift")

axes[1].axhline(0.0, color="0.5", lw=1)
axes[1].plot(jackknife_results["omitted_bin_center"], 1000.0 * jackknife_results["mass_mean_shift"], "o-")
axes[1].set_xlabel(r"omitted mass-bin center (GeV)")
axes[1].set_ylabel("mass mean shift (MeV)")
fig.tight_layout()

### Discussion question 3

If omitting the mass bin at the peak shifts the signal yield strongly, is that a problem with the fit, expected behavior, or both? What would be more concerning: a large shift near the peak or a large shift far away in the sideband?

# Part 8: Comparing uncertainty estimates

The table below compares uncertainty estimates from three ideas:

- likelihood curvature: local quadratic approximation,
- bootstrap: repeated fake experiments generated from the fitted intensity,
- jackknife: sensitivity to removing mass regions.

They do not answer identical questions, so disagreement is a diagnostic, not just an inconvenience.

In [ ]:
comparison_parameters = ["signal_yield", "mass_mean_GeV", "mass_sigma_GeV"]
uncertainty_comparison = covariance_summary.set_index("parameter").loc[comparison_parameters, ["estimate", "covariance_uncertainty"]].reset_index()

bootstrap_std = bootstrap_results[comparison_parameters].std().rename("bootstrap_std").reset_index().rename(columns={"index": "parameter"})
jackknife_max_shift = (
    jackknife_results[["signal_yield_shift", "mass_mean_shift"]]
    .abs()
    .max()
    .rename(index={"signal_yield_shift": "signal_yield", "mass_mean_shift": "mass_mean_GeV"})
    .rename("max_leave_bin_out_shift")
    .reset_index()
    .rename(columns={"index": "parameter"})
)

uncertainty_comparison = uncertainty_comparison.merge(bootstrap_std, on="parameter", how="left")
uncertainty_comparison = uncertainty_comparison.merge(jackknife_max_shift, on="parameter", how="left")
uncertainty_comparison

# Part 9: Takeaways

- In an event-wise Poisson model, $\lambda(x;\theta)$ is an event intensity, and $\int\lambda\,dx$ is the expected number of events.
- The extended likelihood uses both the total event count and the locations of the events in $(m,p_T)$.
- A likelihood-ratio statistic compares nested intensity models, but boundary cases require care.
- Bootstrap resampling estimates the spread from repeated experiments under an assumed model.
- Jackknife diagnostics show which data regions have high influence on a fitted result.
- Covariance, bootstrap, and jackknife uncertainties are complementary; they should be compared, not blindly averaged.

# References and Further Reading

- D. W. Hogg, J. Bovy, and D. Lang, **Data analysis recipes: Fitting a model to data**, arXiv:1008.4686: https://arxiv.org/abs/1008.4686. See also the course [Unit 2 resources](../resources/unit-2-data-analysis.md).
- T. McCauley, **Dimuon event information derived from the Run2010B public Mu dataset**, CERN Open Data Portal (2014), DOI: [10.7483/OPENDATA.CMS.CB8H.MFFA](https://doi.org/10.7483/OPENDATA.CMS.CB8H.MFFA).
- ROOT project, **RDataFrame CSV data-source tutorial**, using the CMS Run2010B dimuon CSV: https://root.cern/doc/master/df014__CSVDataSource_8C.html
- R. Barlow, **Extended maximum likelihood**, *Nucl. Instrum. Meth. A* 297, 496-506 (1990), DOI: [10.1016/0168-9002(90)91334-8](https://doi.org/10.1016/0168-9002(90)91334-8).
- Particle Data Group, **Statistics review**, in *Review of Particle Physics*: https://pdg.lbl.gov/
- B. Efron and R. Tibshirani, **Bootstrap Methods for Standard Errors, Confidence Intervals, and Other Measures of Statistical Accuracy**, *Statistical Science* 1, 54-75 (1986), DOI: [10.1214/ss/1177013815](https://doi.org/10.1214/ss/1177013815).
- B. Efron, **The Jackknife, the Bootstrap and Other Resampling Plans**, SIAM (1982), DOI: [10.1137/1.9781611970319](https://doi.org/10.1137/1.9781611970319).